# 🚌 Predicting Public Transport Delays from Weather & City Events

**Goal:** predict a trip's arrival delay (minutes) from external factors: weather, city events, traffic and time.

**Dataset:** [Public Transport Delays with Weather & Events (Kaggle)](https://www.kaggle.com/datasets/khushikyad001/public-transport-delays-with-weather-and-events): 2,000 simulated trips, 25 columns.

| Section | Skill practised |
|---|---|
| 1. Load & inspect | — |
| 2. Cleaning | Data cleaning |
| 3. Feature engineering | Feature engineering |
| 4. Weather effects (Q1) | EDA + hypothesis tests |
| 5. Event effects (Q2) | EDA + hypothesis tests |
| 6. Time patterns | Time-based analysis |
| 7. Models | Regression (Linear, Random Forest, XGBoost) |
| 8. Interpretation & prediction | — |

> Put `public_transport_delays.csv` in `data/`. If it isn't there, the notebook falls back to a synthetic sample with the same structure.

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")
# make `src` importable whether the notebook is run from notebooks/ or the project root
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, ROOT); os.chdir(ROOT)

import numpy as np, pandas as pd
from IPython.display import display, Image
from src.data_prep import load_raw, detect_roles, clean
from src.features import add_features, model_columns
from src import eda
from src.modeling import (build_models, evaluate_models, time_split, tune, importance,
                          diagnostic_plots, save_model, metrics, HAS_XGB)
FIG = "outputs/figures"
pd.set_option("display.max_columns", 50)
print("XGBoost available:", HAS_XGB)

## 1. Load & inspect

In [ ]:
DATA = "data/public_transport_delays.csv"
if not os.path.exists(DATA):
    print("⚠️ Real CSV not found, using synthetic sample data")
    DATA = "data/sample_public_transport_delays.csv"
    if not os.path.exists(DATA):
        os.system(f'"{sys.executable}" data/make_sample_data.py')

raw = load_raw(DATA)          # column names normalised to snake_case
print(raw.shape)
display(raw.head())

In [ ]:
# Map each column to a standard role so the rest of the code doesn't depend on exact header names
roles = detect_roles(list(raw.columns))
pd.Series(roles, name="column")

In [ ]:
display(raw.dtypes.to_frame("dtype").T)
print("Missing values per column:")
display(raw.isna().sum()[raw.isna().sum() > 0].to_frame("missing"))
print("Duplicate rows:", raw.duplicated().sum())

## 2. Data cleaning
Steps in `src/data_prep.clean`:
1. Drop duplicate rows and rows with no target value.
2. Parse dates/times, convert Yes/No flags to 0/1.
3. Coerce numeric columns, clip impossible values (humidity 0–100, non-negative wind/rain), impute the median.
4. Tidy categorical text and fill gaps (`event_type` → `"None"`, others → `"Unknown"`).
5. Winsorise the most extreme 1% of delays so a few outliers don't dominate the regression loss.

In [ ]:
df = clean(raw, roles)
TARGET = roles["target"]
df[TARGET].describe()

## 3. Feature engineering
* **Time:** hour, cyclical time-of-day (sin/cos), day of week, weekend, month, time band, scheduled trip duration
* **Weather:** `is_rain`, `is_snow`, `is_fog`, `bad_weather`, `extreme_temp` (≤0 °C or ≥32 °C), distance from a comfortable 18 °C, `heavy_precip`, `high_wind`
* **Events:** `has_event`, `log_attendance`
* **Interactions:** event × peak hour, bad weather × congestion, bad weather × peak

**Leakage check:** `actual_departure_delay_min` and the `delayed` flag are dropped from the model inputs. Both are only known once the trip is underway (or are derived from the target), so including them would inflate scores without helping real forecasting.

In [ ]:
df = add_features(df, roles)
num_feats, cat_feats = model_columns(df, roles, include_departure_delay=False)
print(f"{len(num_feats)} numeric:", num_feats)
print(f"{len(cat_feats)} categorical:", cat_feats)

## 4. Delay overview

In [ ]:
display(eda.target_distribution(df, roles, FIG).to_frame("delay_min"))
Image(f"{FIG}/01_target_distribution.png")

## 5. Q1: How do rain, snow and extreme temperatures affect delays?
A Kruskal–Wallis test checks whether delays differ across weather conditions. It's non-parametric, so it doesn't assume normally distributed delays. Mann–Whitney tests then compare each adverse condition with all other trips.

In [ ]:
wx = eda.weather_effect(df, roles, FIG)
display(wx.get("weather_table"))
print(f"Kruskal-Wallis p-value: {wx.get('weather_kruskal_p', float('nan')):.3g}")
display(wx.get("weather_flags"))
display(wx.get("temperature_table"))
Image(f"{FIG}/02_weather_effect.png")

**Interpretation:** weather makes **no meaningful difference** in this dataset.
* Mean delay is about 13 min under every condition (Clear 13.5, Rain 13.6, Snow 13.5, Storm 13.3 min). Kruskal–Wallis gives p = 0.90, so there's no significant difference.
* Rain (+0.2 min), snow (+0.3 min) and extreme temperatures (−0.2 min) all shift delays by less than half a minute, and none is significant (all p > 0.18).
* There's no U-shape for temperature: every band sits between 12.6 and 13.9 min, and the precipitation trend line is flat.

In real transit systems, rain and snow usually *do* increase delays. Getting no effect here is the first sign that the data was generated at random.

## 6. Q2: Do major city events increase delay probability?
A trip counts as *delayed* if it arrives more than 5 minutes late. A chi-square test checks whether the delay rate depends on an event being nearby.

In [ ]:
ev = eda.event_effect(df, roles, FIG)
for k in ("delay_prob_event", "delay_prob_no_event", "relative_risk", "event_chi2_p"):
    if k in ev: print(f"{k:>22}: {ev[k]:.3g}")
display(ev.get("event_table"))
Image(f"{FIG}/03_event_effect.png")

**Interpretation:** city events **don't increase delay probability** here.
* With an event nearby, 74.1% of trips are delayed by more than 5 min. Without one, 75.5% are. The relative risk is 0.98× (chi-square p = 0.51).
* No event type stands out: every type averages about 12.8–13.9 min.
* Event size shows only a tiny correlation (ρ ≈ 0.05). With 15+ tests run, that's consistent with chance.

## 7. Time-based analysis

In [ ]:
tp = eda.time_patterns(df, roles, FIG)
for k in ("peak", "season", "transport"):
    if k in tp: display(tp[k].to_frame(f"mean delay by {k}"))
Image(f"{FIG}/04_time_patterns.png")

In [ ]:
display(eda.correlations(df, roles, FIG).to_frame("spearman_rho"))
Image(f"{FIG}/05_correlations.png")

## 8. Modelling
* **Split:** chronological. The model trains on the earliest 80% of trips and is tested on the most recent 20%, the same way a deployed model would forecast future trips.
* **Validation:** 5-fold CV on the training set.
* **Models:** mean baseline → Linear / Ridge Regression → Random Forest → XGBoost (falls back to sklearn `HistGradientBoostingRegressor` if `xgboost` isn't installed).
* **Metrics:** MAE (average minutes off), RMSE (penalises big misses), R².

In [ ]:
train_idx, test_idx = time_split(df, roles, test_frac=0.2)
X, y = df[num_feats + cat_feats], df[TARGET]
X_tr, X_te, y_tr, y_te = X.loc[train_idx], X.loc[test_idx], y.loc[train_idx], y.loc[test_idx]
print("train:", X_tr.shape, " test:", X_te.shape)

results, fitted = evaluate_models(build_models(num_feats, cat_feats), X_tr, y_tr, X_te, y_te)
results.sort_values("Test_MAE")

In [ ]:
# Tune the best tree-based model with a randomized search
tree_models = [m for m in results.index if m not in ("Baseline (mean)", "Linear Regression", "Ridge Regression")]
tree_best = results.loc[tree_models].sort_values("CV_MAE").index[0]
tuned, params = tune(fitted[tree_best], X_tr, y_tr, n_iter=20)
tm = metrics(y_te, tuned.predict(X_te))
print(tree_best, "tuned params:", params)
print({k: round(v, 3) for k, v in tm.items()})
results.loc[f"{tree_best} (tuned)"] = [np.nan, np.nan, np.nan, tm["MAE"], tm["RMSE"], tm["R2"]]
results.round(3).sort_values("Test_MAE")

In [ ]:
best_name = results.drop("Baseline (mean)").sort_values("Test_MAE").index[0]
best = tuned if best_name.endswith("(tuned)") else fitted[best_name]
y_pred = best.predict(X_te)
diagnostic_plots(y_te, y_pred, results, FIG, best_name)
print("Best model:", best_name)
Image(f"{FIG}/06_model_results.png")

## 9. What drives delays? (permutation importance)
Each feature is shuffled in turn. The bigger the increase in MAE, the more the model relies on it.

In [ ]:
imp = importance(best, X_te, y_te, FIG)
display(imp.head(15))
Image(f"{FIG}/07_feature_importance.png")

## 10. Save the model & predict new trips

In [ ]:
save_model(best, dict(model_name=best_name, numeric_features=num_feats,
                      categorical_features=cat_feats, roles=roles), "outputs/models/best_model.joblib")
from predict import predict
scenario = pd.DataFrame([
    dict(date="2024-11-15", time="17:30:00", transport_type="Bus", route_id="Route_7",
         scheduled_arrival="18:05:00", weather_condition=w, temperature_c=t, precipitation_mm=p,
         wind_speed_kmh=wd, event_type=e, event_attendance_est=a, traffic_congestion_index=c,
         humidity_percent=65, holiday=0, peak_hour=1, weekday="Friday", season="Autumn")
    for w, t, p, wd, e, a, c in [("Clear", 14, 0, 10, "None", 0, 50),
                                 ("Rain", 9, 8, 20, "None", 0, 65),
                                 ("Snow", -3, 6, 25, "None", 0, 70),
                                 ("Storm", 12, 15, 45, "Sports", 45000, 85)]])
pred, _ = predict(scenario)
scenario.assign(predicted_delay_min=pred.round(1))[["weather_condition", "event_type", "traffic_congestion_index", "predicted_delay_min"]]

## 11. Conclusions
1. **Weather:** rain, snow, fog, wind and extreme temperatures don't change delays. Every condition averages about 13 min (p = 0.90).
2. **Events:** trips near concerts, sports, protests, festivals or parades are no more likely to be delayed (74.1% vs 75.5%, p = 0.51).
3. **Time:** peak and off-peak hours (13.4 vs 13.3 min), weekdays, seasons and transport modes are all essentially identical.
4. **Models:** no model beats the mean baseline. Baseline test MAE is 8.16 min. Random Forest scores 8.10, Ridge 8.15, Linear Regression 8.19 and Gradient Boosting 8.23, with R² ≈ 0 for all of them. Permutation importance is close to zero for every feature, so the models found nothing to learn.

**Why?** The dataset is synthetic, and each column appears to have been generated independently:
* Departure delay and arrival delay are uncorrelated (ρ = −0.006). For real trips they would be strongly linked.
* Season labels don't match the dates (e.g. "Autumn" in January).
* Some trips with `event_type = None` still have an attendance value.

**What this project shows:** the full workflow (cleaning, feature engineering, hypothesis tests, a leakage-free chronological split, cross-validation, and model comparison against a baseline) was applied correctly. It also showed that **the data contains no learnable signal**. Comparing against a baseline and testing for significance stopped us from reporting a model that only looks good on paper. On the synthetic demo file with built-in effects (`data/sample_public_transport_delays.csv`), the same pipeline gets R² ≈ 0.75, which confirms the method works when a signal exists.

**Limitations & next steps**
* The Kaggle data is *synthetic*, so real-world effects may differ. Validate on a real feed (e.g., GTFS-realtime + a weather API).
* Try LightGBM/CatBoost, quantile regression for prediction intervals, or a classifier for P(delay > 5 min).
* Add lag features (the previous trip's delay on the same route) if you have consecutive trips.